# Unit 07 — Pandas for Data Manipulation
### Stage 1: Core Foundations | Data Science & AI Program

**Business scenario:** AdventureWorks — a fictional e-commerce company selling electronics, home products, and accessories.

**Audience:** Learners who completed Units 01–06  
**Environment:** Python 3 · Google Colab / Jupyter Notebook  
**Data policy:** Small, deterministic synthetic datasets created in the notebook. No external files, API keys, or downloads are required.

> **Instructor note:** Unit 06 introduced NumPy and Pandas. This unit deepens practical Pandas skills: working with Series and DataFrames, reading common business files, cleaning data, transforming columns, combining tables, and producing summaries. Keep the examples consistent with the AdventureWorks business context.

## Learning outcomes

By the end of this unit, you should be able to:
- Explain the difference between a Pandas `Series` and a `DataFrame`.
- Read and write CSV, JSON, and Excel files, understanding Excel's optional dependency.
- Inspect DataFrames with `head()`, `info()`, `describe()`, `shape`, and `dtypes`.
- Select, filter, sort, and transform rows and columns.
- Identify and handle missing values, duplicates, and incorrect data types.
- Create new columns using vectorized operations and business rules.
- Summarize records using `groupby()` and aggregation.
- Combine related tables using `merge()` and `concat()`.
- Export a cleaned dataset for later analysis.
- Explain how these operations support EDA, feature engineering, and ML pipelines.

### Prerequisites
Python collections, loops, functions, file handling, and Unit 06 NumPy/Pandas basics.

## Table of contents
1. AdventureWorks data model
2. Series and DataFrame
3. Inspecting data
4. Reading CSV, JSON, and Excel
5. Selecting, filtering, and sorting
6. Creating and transforming columns
7. Missing values, duplicates, and data types
8. Grouping and aggregation
9. Merging related tables
10. Concatenating datasets
11. Integrated data preparation workflow
12. Student challenge


## 1. AdventureWorks data model

AdventureWorks wants to prepare a clean order dataset for reporting. We will use two related tables:
- **Order lines:** one row per product line in an order.
- **Product catalogue:** one row per product, including its category and supplier.

We will also create a small second-period order table to demonstrate concatenation. All values are synthetic and designed for learning.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import json

orders_data = {
    "order_id": ["AW-5101", "AW-5102", "AW-5103", "AW-5104", "AW-5105", "AW-5106", "AW-5107"],
    "order_date": ["2026-09-01", "2026-09-02", "2026-09-02", "2026-09-03", "2026-09-04", "2026-09-04", "2026-09-05"],
    "customer": ["Maya Chen", "Omar Saleh", "Lina Haddad", "Noah Reed", "Maya Chen", "Sara Nasser", "Omar Saleh"],
    "product_id": ["P-101", "P-103", "P-104", "P-102", "P-105", "P-104", "P-106"],
    "quantity": [2, 1, 2, 1, 1, 3, 2],
    "unit_price": [24.99, 79.50, 28.75, 34.00, 59.99, 28.75, 39.00],
    "status": ["Shipped", "Processing", "Shipped", "Cancelled", "Processing", "Shipped", "Processing"],
    "payment_method": ["Card", "Wallet", "Card", "Card", "Wallet", "Card", "Wallet"]
}
orders = pd.DataFrame(orders_data)

products = pd.DataFrame({
    "product_id": ["P-101", "P-102", "P-103", "P-104", "P-105", "P-106"],
    "product_name": ["Wireless Mouse", "USB-C Hub", "Mechanical Keyboard", "Desk Lamp", "Headphones", "Laptop Stand"],
    "category": ["Accessories", "Accessories", "Electronics", "Home", "Electronics", "Accessories"],
    "supplier": ["Northstar Supply", "Northstar Supply", "KeyWorks", "BrightHome", "SoundSphere", "Northstar Supply"],
    "list_price": [24.99, 34.00, 79.50, 28.75, 59.99, 39.00]
})

orders_period_2 = pd.DataFrame({
    "order_id": ["AW-5108", "AW-5109"],
    "order_date": ["2026-09-06", "2026-09-07"],
    "customer": ["Rami Khalil", "Maya Chen"],
    "product_id": ["P-101", "P-105"],
    "quantity": [1, 2],
    "unit_price": [24.99, 59.99],
    "status": ["Shipped", "Processing"],
    "payment_method": ["Card", "Wallet"]
})

display(orders.head())
display(products.head())

,order_id,order_date,customer,product_id,quantity,unit_price,status,payment_method
0,AW-5101,2026-09-01,Maya Chen,P-101,2,24.99,Shipped,Card
1,AW-5102,2026-09-02,Omar Saleh,P-103,1,79.50,Processing,Wallet
2,AW-5103,2026-09-02,Lina Haddad,P-104,2,28.75,Shipped,Card
3,AW-5104,2026-09-03,Noah Reed,P-102,1,34.00,Cancelled,Card
4,AW-5105,2026-09-04,Maya Chen,P-105,1,59.99,Processing,Wallet


,product_id,product_name,category,supplier,list_price
0,P-101,Wireless Mouse,Accessories,Northstar Supply,24.99
1,P-102,USB-C Hub,Accessories,Northstar Supply,34.00
2,P-103,Mechanical Keyboard,Electronics,KeyWorks,79.50
3,P-104,Desk Lamp,Home,BrightHome,28.75
4,P-105,Headphones,Electronics,SoundSphere,59.99


### Understand the table grain

The `orders` table is at **order-line level** in this teaching dataset: each row represents one product line. In a production system, an order can contain several lines, so a robust line-level key often needs both an order ID and a line ID. Here, the data is simplified to keep the focus on Pandas.

## 2. Series and DataFrame

A **Series** is a one-dimensional labeled array. A **DataFrame** is a two-dimensional table whose columns are Series.

Select one column to get a Series, or select a list of columns to get a DataFrame.

In [2]:
customer_series = orders["customer"]
order_view = orders[["order_id", "customer", "status"]]

print("Single column type:", type(customer_series))
print("Multiple columns type:", type(order_view))
display(customer_series.head())
display(order_view.head())

Single column type: <class 'pandas.core.series.Series'>
Multiple columns type: <class 'pandas.core.frame.DataFrame'>


,customer
0,Maya Chen
1,Omar Saleh
2,Lina Haddad
3,Noah Reed
4,Maya Chen


,order_id,customer,status
0,AW-5101,Maya Chen,Shipped
1,AW-5102,Omar Saleh,Processing
2,AW-5103,Lina Haddad,Shipped
3,AW-5104,Noah Reed,Cancelled
4,AW-5105,Maya Chen,Processing


**Teaching point:** The brackets matter: `orders["customer"]` returns a Series; `orders[["customer"]]` returns a one-column DataFrame.

**Exercise 2A:** Select `unit_price` as a Series, then select `unit_price` and `quantity` as a DataFrame. Print each object's type.

## 3. Inspecting data

Inspect a dataset before analysis. This helps reveal unexpected data types, missing values, unusual values, and the number of records.

In [3]:
print("Shape:", orders.shape)
print("Columns:", orders.columns.tolist())
print("\nData types:")
print(orders.dtypes)
display(orders.head(3))
display(orders.tail(2))

Shape: (7, 8)
Columns: ['order_id', 'order_date', 'customer', 'product_id', 'quantity', 'unit_price', 'status', 'payment_method']

Data types:
order_id           object
order_date         object
customer           object
product_id         object
quantity            int64
unit_price        float64
status             object
payment_method     object
dtype: object


,order_id,order_date,customer,product_id,quantity,unit_price,status,payment_method
0,AW-5101,2026-09-01,Maya Chen,P-101,2,24.99,Shipped,Card
1,AW-5102,2026-09-02,Omar Saleh,P-103,1,79.50,Processing,Wallet
2,AW-5103,2026-09-02,Lina Haddad,P-104,2,28.75,Shipped,Card


,order_id,order_date,customer,product_id,quantity,unit_price,status,payment_method
5,AW-5106,2026-09-04,Sara Nasser,P-104,3,28.75,Shipped,Card
6,AW-5107,2026-09-05,Omar Saleh,P-106,2,39.00,Processing,Wallet


In [ ]:
orders.info()

In [6]:
display(orders.describe(include="all"))

,order_id,order_date,customer,product_id,quantity,unit_price,status,payment_method
count,7,7,7,7,7.000000,7.000000,7,7
unique,7,5,5,6,NaN,NaN,3,2
top,AW-5101,2026-09-02,Maya Chen,P-104,NaN,NaN,Shipped,Card
freq,1,2,2,2,NaN,NaN,3,4
mean,NaN,NaN,NaN,NaN,1.714286,42.140000,NaN,NaN
std,NaN,NaN,NaN,NaN,0.755929,20.179272,NaN,NaN
min,NaN,NaN,NaN,NaN,1.000000,24.990000,NaN,NaN
25%,NaN,NaN,NaN,NaN,1.000000,28.750000,NaN,NaN
50%,NaN,NaN,NaN,NaN,2.000000,34.000000,NaN,NaN
75%,NaN,NaN,NaN,NaN,2.000000,49.495000,NaN,NaN


`head()` and `tail()` preview records. `info()` reports non-null counts and data types. `describe()` summarizes columns; `include="all"` requests summaries for both numeric and non-numeric columns, although different types have different statistics.

**Exercise 3A:** Identify which columns are numeric, which are text, and which should represent dates. Is `order_date` currently stored as a date type?

## 4. Reading and writing CSV, JSON, and Excel

Pandas supports multiple tabular formats. Common readers include `read_csv()`, `read_json()`, and `read_excel()`. Writer methods include `to_csv()`, `to_json()`, and `to_excel()`.

CSV is simple and widely supported. JSON is useful for nested or API-style data. Excel is common in business workflows and may require an optional engine such as `openpyxl`.

### 4.1 CSV

We will write a file inside a local folder, then load it back. No external download is needed.

In [7]:
data_folder = Path("adventureworks_data")
data_folder.mkdir(exist_ok=True)
csv_path = data_folder / "orders_unit07.csv"

orders.to_csv(csv_path, index=False)
orders_from_csv = pd.read_csv(csv_path)

print("Saved:", csv_path)
print("Loaded shape:", orders_from_csv.shape)
display(orders_from_csv.head(2))

Saved: adventureworks_data/orders_unit07.csv
Loaded shape: (7, 8)


,order_id,order_date,customer,product_id,quantity,unit_price,status,payment_method
0,AW-5101,2026-09-01,Maya Chen,P-101,2,24.99,Shipped,Card
1,AW-5102,2026-09-02,Omar Saleh,P-103,1,79.50,Processing,Wallet


### 4.2 JSON

For this example, use the `records` orientation: each row becomes a JSON object in a list.

In [8]:
json_path = data_folder / "orders_unit07.json"
orders.to_json(json_path, orient="records", indent=2)
orders_from_json = pd.read_json(json_path)

print("Loaded JSON shape:", orders_from_json.shape)
display(orders_from_json.head(2))

Loaded JSON shape: (7, 8)


,order_id,order_date,customer,product_id,quantity,unit_price,status,payment_method
0,AW-5101,2026-09-01,Maya Chen,P-101,2,24.99,Shipped,Card
1,AW-5102,2026-09-02,Omar Saleh,P-103,1,79.50,Processing,Wallet


### 4.3 Excel

Excel support may require installing an optional reader/writer engine. In Google Colab, `openpyxl` is commonly available; if it is missing, install it with `%pip install openpyxl` in a separate setup cell. The following code is guarded so the notebook can continue if the engine is unavailable.

In [9]:
excel_path = data_folder / "orders_unit07.xlsx"

try:
    orders.to_excel(excel_path, index=False, sheet_name="OrderLines")
    orders_from_excel = pd.read_excel(excel_path, sheet_name="OrderLines")
    print("Excel round-trip shape:", orders_from_excel.shape)
except ImportError as error:
    print("Excel engine is not installed in this environment.")
    print("In Colab, run: %pip install openpyxl")
    print("Details:", error)

Excel round-trip shape: (7, 8)


**Important:** After reading any file, inspect the resulting DataFrame. Dates, numeric columns, missing-value markers, and identifiers may need explicit parsing or validation.

**Exercise 4A:** Export `products` to CSV, load it back, and verify that its row count is unchanged.

## 5. Selecting, filtering, and sorting

Select columns by name, rows by index, and records by conditions. Use parentheses around each condition when combining conditions with `&` or `|`.

In [ ]:
display(orders[["order_id", "customer", "status"]].head())

processing = orders[orders["status"] == "Processing"]
display(processing)

higher_value_lines = orders[
    (orders["quantity"] >= 2) &
    (orders["unit_price"] >= 30)
]
display(higher_value_lines[["order_id", "product_id", "quantity", "unit_price"]])

### Sorting

Use `sort_values()` to order records. Sorting helps prioritize records for review; it does not change the underlying business meaning.

In [ ]:
sorted_orders = orders.sort_values(
    by=["status", "unit_price"],
    ascending=[True, False]
)
display(sorted_orders[["order_id", "status", "unit_price"]])

**Exercise 5A:** Display only shipped order lines, sorted by `unit_price` from highest to lowest.

## 6. Creating and transforming columns

Vectorized column operations are generally clearer than looping over DataFrame rows. We'll calculate line value and normalize customer names using string methods.

In [10]:
orders_work = orders.copy()
orders_work["line_total"] = orders_work["quantity"] * orders_work["unit_price"]
orders_work["customer_clean"] = orders_work["customer"].str.strip().str.title()

display(orders_work[["order_id", "customer", "customer_clean", "quantity", "unit_price", "line_total"]])

,order_id,customer,customer_clean,quantity,unit_price,line_total
0,AW-5101,Maya Chen,Maya Chen,2,24.99,49.98
1,AW-5102,Omar Saleh,Omar Saleh,1,79.50,79.50
2,AW-5103,Lina Haddad,Lina Haddad,2,28.75,57.50
3,AW-5104,Noah Reed,Noah Reed,1,34.00,34.00
4,AW-5105,Maya Chen,Maya Chen,1,59.99,59.99
5,AW-5106,Sara Nasser,Sara Nasser,3,28.75,86.25
6,AW-5107,Omar Saleh,Omar Saleh,2,39.00,78.00


### Convert data types

`order_date` was read as text. Convert it to a datetime type with `pd.to_datetime()`. For data that may contain invalid date strings, `errors="coerce"` turns invalid values into `NaT`, which can then be investigated.

In [11]:
orders_work["order_date"] = pd.to_datetime(orders_work["order_date"], errors="coerce")
print(orders_work["order_date"].dtype)
display(orders_work[["order_id", "order_date"]])

datetime64[ns]


,order_id,order_date
0,AW-5101,2026-09-01
1,AW-5102,2026-09-02
2,AW-5103,2026-09-02
3,AW-5104,2026-09-03
4,AW-5105,2026-09-04
5,AW-5106,2026-09-04
6,AW-5107,2026-09-05


### Create a rule-based feature

Create a simple label for order lines with a value of at least `$100`. This is an example feature, not an official AdventureWorks business policy.

In [12]:
orders_work["line_value_band"] = np.where(
    orders_work["line_total"] >= 100,
    "High",
    "Standard"
)
display(orders_work[["order_id", "line_total", "line_value_band"]])

,order_id,line_total,line_value_band
0,AW-5101,49.98,Standard
1,AW-5102,79.50,Standard
2,AW-5103,57.50,Standard
3,AW-5104,34.00,Standard
4,AW-5105,59.99,Standard
5,AW-5106,86.25,Standard
6,AW-5107,78.00,Standard


**Exercise 6A:** Add a `is_multi_unit` column that is `True` when quantity is greater than one. Create a second column that shows the month of `order_date`.

## 7. Missing values, duplicates, and data types

Data quality work should begin with measurement and investigation. Do not automatically remove or fill values without understanding the business context.

In [13]:
quality = orders_work.copy()
quality.loc[1, "customer"] = None
quality.loc[2, "unit_price"] = np.nan

print("Missing values per column:")
print(quality.isna().sum())

Missing values per column:
order_id           0
order_date         0
customer           1
product_id         0
quantity           0
unit_price         1
status             0
payment_method     0
line_total         0
customer_clean     0
line_value_band    0
dtype: int64


### Handle missing values

For this teaching example, fill a missing customer label with `"Unknown"` for reporting, and leave the missing price unresolved until it can be checked against a trusted product catalogue. Filling a missing price with zero would incorrectly imply that the product was free.

In [14]:
quality["customer"] = quality["customer"].fillna("Unknown")
display(quality.loc[[1, 2], ["order_id", "customer", "unit_price"]])

,order_id,customer,unit_price
1,AW-5102,Unknown,79.5
2,AW-5103,Lina Haddad,NaN


### Detect and remove exact duplicate rows

`duplicated()` flags duplicate rows; `drop_duplicates()` removes exact duplicates. In production, determine the correct business key first. Two lines can share an order ID while representing different products.

In [15]:
duplicate_rows = quality[quality.duplicated(keep=False)]
print("Exact duplicate rows flagged:", len(duplicate_rows))

quality_deduplicated = quality.drop_duplicates().copy()
print("Rows before:", len(quality))
print("Rows after:", len(quality_deduplicated))

Exact duplicate rows flagged: 0
Rows before: 7
Rows after: 7


### Validate and convert types

Use `astype()` when the values are already compatible with the target type. Use `pd.to_numeric()` or `pd.to_datetime()` when parsing may be required. Avoid converting identifiers to numbers just because they contain digits: IDs are labels, not quantities.

In [ ]:
# Example: safely parse a numeric column if values may contain invalid text
price_text = pd.Series(["24.99", "34.00", "not available"])
parsed_prices = pd.to_numeric(price_text, errors="coerce")

print(parsed_prices)
print("Values that failed conversion:", parsed_prices.isna().sum())

**Exercise 7A:** Make a copy of `orders`, insert one missing `status`, and count missing values. Decide whether to fill it with `"Unknown"` or leave it missing, and explain why. Also identify any exact duplicate rows.

## 8. Grouping and aggregation

`groupby()` follows a split–apply–combine pattern: split rows into groups, calculate metrics, and combine the results. The `agg()` method lets us name multiple summary metrics clearly.

In [16]:
orders_work["line_total"] = orders_work["quantity"] * orders_work["unit_price"]

category_summary = (
    orders_work.groupby("product_id", as_index=False)
    .agg(
        line_count=("order_id", "count"),
        units_sold=("quantity", "sum"),
        total_line_value=("line_total", "sum"),
        average_unit_price=("unit_price", "mean")
    )
    .sort_values("total_line_value", ascending=False)
)

display(category_summary)

,product_id,line_count,units_sold,total_line_value,average_unit_price
3,P-104,2,5,143.75,28.75
2,P-103,1,1,79.50,79.50
5,P-106,1,2,78.00,39.00
4,P-105,1,1,59.99,59.99
0,P-101,1,2,49.98,24.99
1,P-102,1,1,34.00,34.00


The output summarizes each product ID. Because this small dataset includes processing and cancelled records, `total_line_value` is not official realized revenue. A business report must define which statuses count and how refunds, discounts, and taxes are handled.

In [17]:
status_summary = (
    orders_work.groupby("status", as_index=False)
    .agg(
        line_count=("order_id", "count"),
        units=("quantity", "sum"),
        line_value=("line_total", "sum")
    )
    .sort_values("line_count", ascending=False)
)
display(status_summary)

,status,line_count,units,line_value
1,Processing,3,4,217.49
2,Shipped,3,7,193.73
0,Cancelled,1,1,34.00


**Exercise 8A:** Group by `payment_method` and calculate the number of lines and total line value for each payment method.

## 9. Merging related tables

Use `merge()` to combine tables using a shared key, similar to a database join. Here, `orders_work` contains `product_id`, while `products` provides the product name, category, and supplier.

Before merging, check whether the lookup key is unique in the product table. Duplicate keys can multiply rows unexpectedly.

In [18]:
print("Product IDs unique:", products["product_id"].is_unique)

orders_enriched = orders_work.merge(
    products[["product_id", "product_name", "category", "supplier"]],
    on="product_id",
    how="left",
    validate="many_to_one"
)

display(orders_enriched[[
    "order_id", "product_id", "product_name", "category", "supplier", "line_total"
]])

Product IDs unique: True


,order_id,product_id,product_name,category,supplier,line_total
0,AW-5101,P-101,Wireless Mouse,Accessories,Northstar Supply,49.98
1,AW-5102,P-103,Mechanical Keyboard,Electronics,KeyWorks,79.50
2,AW-5103,P-104,Desk Lamp,Home,BrightHome,57.50
3,AW-5104,P-102,USB-C Hub,Accessories,Northstar Supply,34.00
4,AW-5105,P-105,Headphones,Electronics,SoundSphere,59.99
5,AW-5106,P-104,Desk Lamp,Home,BrightHome,86.25
6,AW-5107,P-106,Laptop Stand,Accessories,Northstar Supply,78.00


### Join types

- `how="left"` keeps every row from the left table and adds matching information from the right.
- `how="inner"` keeps only rows with a match in both tables.
- `how="right"` keeps every row from the right table.
- `how="outer"` keeps all keys from both tables.

The `validate="many_to_one"` argument checks that many order rows can match one product row. It helps catch unexpected duplicate product keys.

In [ ]:
# Find order lines whose product ID is not present in the catalogue
unmatched_products = orders_work[
    ~orders_work["product_id"].isin(products["product_id"])
]
display(unmatched_products[["order_id", "product_id"]])

**Exercise 9A:** Merge orders with products to include `product_name` and `category`. Then count how many order lines did not find a matching product.

## 10. Concatenating datasets

`concat()` stacks compatible DataFrames by rows or joins them by columns. It is useful when two files have the same schema but represent different periods.

Unlike `merge()`, `concat()` does not match records using a business key.

In [ ]:
all_orders = pd.concat([orders, orders_period_2], ignore_index=True)

print("Original period 1 rows:", len(orders))
print("Period 2 rows:", len(orders_period_2))
print("Combined rows:", len(all_orders))
display(all_orders.tail(3))

`ignore_index=True` creates a fresh sequential index for the combined table. If the column names differ between inputs, Pandas aligns by column name and may create missing values for columns not present in one input.

**Exercise 10A:** Create another two-row DataFrame with the same columns as `orders_period_2`, concatenate it, and confirm the expected total row count.

## 11. Integrated AdventureWorks data preparation workflow

This workflow combines the unit's key skills: parse dates, calculate line values, merge product details, filter records, and create a category summary.

In [ ]:
# Start from a clean copy so earlier demonstrations do not affect the workflow
workflow_orders = orders.copy()

workflow_orders["order_date"] = pd.to_datetime(
    workflow_orders["order_date"],
    errors="coerce"
)
workflow_orders["line_total"] = (
    workflow_orders["quantity"] * workflow_orders["unit_price"]
)

workflow_enriched = workflow_orders.merge(
    products[["product_id", "product_name", "category"]],
    on="product_id",
    how="left",
    validate="many_to_one"
)

# Example reporting view: exclude cancelled lines for this summary only
reporting_lines = workflow_enriched[
    workflow_enriched["status"] != "Cancelled"
].copy()

weekly_summary = (
    reporting_lines.groupby("category", as_index=False)
    .agg(
        order_lines=("order_id", "count"),
        units_sold=("quantity", "sum"),
        line_value=("line_total", "sum")
    )
    .sort_values("line_value", ascending=False)
)

display(reporting_lines.head())
display(weekly_summary)

**Interpretation:** This report excludes cancelled lines because the example reporting question focuses on non-cancelled activity. This is a teaching choice, not a universal accounting rule. Real reports must document the exact status and revenue definitions.

In [ ]:
output_folder = Path("adventureworks_data")
output_folder.mkdir(exist_ok=True)
cleaned_path = output_folder / "adventureworks_orders_cleaned.csv"

reporting_lines.to_csv(cleaned_path, index=False)
reloaded_report = pd.read_csv(cleaned_path)

print("Exported file:", cleaned_path)
print("Exported rows:", len(reporting_lines))
print("Reloaded rows:", len(reloaded_report))
print("Columns:", reloaded_report.columns.tolist())

## 12. Student challenge — Prepare a reporting-ready dataset

### Scenario
The AdventureWorks operations team needs a reliable order-line dataset for a weekly review.

### Tasks
1. Inspect the source tables and identify the grain of each table.
2. Convert `order_date` to a datetime type.
3. Calculate `line_total`.
4. Merge product names and categories into the order data.
5. Check for missing values, duplicate rows, and unmatched product IDs.
6. Create a copy with a documented missing-value strategy.
7. Exclude cancelled records from one clearly labelled reporting view.
8. Group by category and calculate line count, units, and line value.
9. Concatenate the two order periods.
10. Export the reporting view to CSV and reload it to verify the result.

### Acceptance criteria
- Use clear, descriptive column names.
- Keep source tables unchanged during cleaning.
- Use `validate="many_to_one"` when merging orders to the product catalogue.
- Explain why `merge()` and `concat()` solve different problems.
- Document how missing values, duplicates, and cancelled orders are handled.
- Do not describe simplified line value as official revenue.

## 13. Instructor answer key

> **Instructor use:** Give learners time to complete the challenge before revealing this reference solution.

In [ ]:
# Reference solution
solution_orders = orders.copy()
solution_orders["order_date"] = pd.to_datetime(solution_orders["order_date"], errors="coerce")
solution_orders["line_total"] = solution_orders["quantity"] * solution_orders["unit_price"]

solution_enriched = solution_orders.merge(
    products[["product_id", "product_name", "category"]],
    on="product_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

unmatched_count = (solution_enriched["_merge"] == "left_only").sum()
solution_enriched = solution_enriched.drop(columns="_merge")

solution_report = solution_enriched[
    solution_enriched["status"] != "Cancelled"
].copy()

solution_summary = (
    solution_report.groupby("category", as_index=False)
    .agg(
        order_lines=("order_id", "count"),
        units_sold=("quantity", "sum"),
        line_value=("line_total", "sum")
    )
    .sort_values("line_value", ascending=False)
)

combined_periods = pd.concat([orders, orders_period_2], ignore_index=True)

output_folder = Path("adventureworks_data")
output_folder.mkdir(exist_ok=True)
solution_file = output_folder / "adventureworks_reporting_solution.csv"
solution_report.to_csv(solution_file, index=False)
verified = pd.read_csv(solution_file)

print("Unmatched product IDs:", unmatched_count)
print("Combined period row count:", len(combined_periods))
print("Export/reload row counts:", len(solution_report), len(verified))
display(solution_summary)